# TSMOM risk-budget diagnostic: current cluster-cap vs. ERC vs. HRP

Wraps `scripts/tsmom_risk_budget_diagnostic.py` -- see that file's module
docstring for the full context (motivated by
`research/cta-layer-separation-risk-budgeting.md`), the library
deviations from the original brief (riskfolio-lib uninstalled after it
broke numpy/pandas project-wide; ERC is a small scipy solver instead;
HRP needs a one-line scipy>=1.18 compatibility shim), and exactly what
each returned field means.

Requires a live IB connection (TWS/IB Gateway running) -- this notebook
doesn't change that, it just calls `main()` directly instead of going
through a real command line, and gives you the returned artifacts
(correlation matrix, weights, etc.) to plot/inspect inline.

In [ ]:
import sys
from pathlib import Path

# scripts/ isn't part of the installed options_bt package (see
# pyproject.toml's [tool.setuptools.packages.find]) -- put the project
# root on sys.path so `from scripts...` resolves regardless of where
# Jupyter's CWD happens to be.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'nbs' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import seaborn as sns

from scripts.tsmom_risk_budget_diagnostic import main

## Params

Same flags as the CLI (`python -m scripts.tsmom_risk_budget_diagnostic --help`
for the full list with descriptions) -- edit this list and re-run the next
cell to try different account sizes, EWM halflives, instrument subsets, etc.
`--account-equity` is the only required one.

In [ ]:
account_equity = 80_000
halflife = 60          # EWM covariance halflife (days) -- see module docstring re: AQR's 60d
min_synced_rows = 252  # require >= 1y of common history across every instrument

argv = [
    '--account-equity', str(account_equity),
    '--halflife', str(halflife),
    '--min-synced-rows', str(min_synced_rows),
    # '--instruments', 'MES,MNQ,MCL,MZL,MZC,MZS,MZW,MGC,SIL,J7,BRE,6M',  # default: full KNOWN_INSTRUMENTS universe
    '--host', '127.0.0.1',
    '--port', '7496',
    '--client-id', '21',
    '--no-save',  # comment out to also write CSVs to results/, as the CLI does by default
]

In [ ]:
# Connects to IB, fetches 3y of continuous front-month bars for every
# instrument, computes ERC/HRP/current-system weights, and returns every
# intermediate artifact -- same console output as the CLI, plus the dict.
results = main(argv)
results.keys()

In [ ]:
report = results['report']
summary = results['summary']
cm = results['corr']  # correlation matrix, sorted by hand-assigned cluster

report

In [ ]:
summary

## Correlation matrix

Sorted by the live system's hand-assigned cluster, then symbol -- if the
clusters are picking up a real factor, same-cluster blocks along the
diagonal should visibly run hotter than cross-cluster cells.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='.2f', cmap='coolwarm', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5, ax=ax)
ax.set_title(f"EWM correlation (halflife={halflife}d), sorted by hand-assigned cluster")
plt.tight_layout()

## Weight comparison: current cluster-cap vs. ERC vs. HRP

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
report[['current_weight', 'erc_weight', 'hrp_weight']].plot.bar(ax=ax)
ax.set_ylabel('Weight (fraction of total |risk|)')
ax.set_title('Risk-budget weight by instrument: current system vs. ERC vs. HRP')
ax.axhline(0, color='black', linewidth=0.8)
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
report[['erc_minus_current', 'hrp_minus_current']].plot.bar(ax=ax)
ax.set_ylabel('Divergence from current weight')
ax.set_title('Where would ERC/HRP size up or down relative to today?')
ax.axhline(0, color='black', linewidth=0.8)
plt.tight_layout()